# GASTAT Labor Market — Data Extraction
**Source:** General Authority for Statistics (GASTAT), 20 quarterly files Q1 2021 – Q4 2025  
**Supplementary:** GOSI open data (sector split) · KAPSARC administrative records (headcount)

| Output CSV | Source | Sheet / File | Scope |
|---|---|---|---|
| `combined_main.csv` | GASTAT | `1` + `6-1` | All 20 files |
| `sector_employment.csv` | GASTAT | `3-6` | Latest file (time series) |
| `private_sector_willingness.csv` | GASTAT | `2-11` | Latest file (time series) |
| `working_hours.csv` | GASTAT | `6-4` | Latest file (time series) |
| `occupation_employment.csv` | GASTAT | `3-3` | All 20 files |
| `economic_activity_employment.csv` | GASTAT | `3-4` | All 20 files |
| `wages_by_sector.csv` | GASTAT | `6-3` | 2024+ files only |
| `gosi_headcount.csv` | KAPSARC | CSV | 2017–2025 quarterly |
| `gosi_sector_split.csv` | GOSI | zip xlsx files | 2025 Q1–2026 Q1 |

In [16]:
import pandas as pd
import os, glob, re
import warnings
warnings.filterwarnings('ignore')

RAW_DIR    = r"C:\Users\Ragha\Projects\gastat-labor-market\data\raw"
OUTPUT_DIR = r"C:\Users\Ragha\Projects\gastat-labor-market\output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

LATEST_FILE = os.path.join(RAW_DIR, 'Labor Market Statistics Q4 2025 AR (1).xlsx')

all_files = sorted(glob.glob(os.path.join(RAW_DIR, '*.xlsx')))
print(f"Found {len(all_files)} files:")
for f in all_files:
    print(f"  {os.path.basename(f)}")
print(f"\nLatest file: {os.path.basename(LATEST_FILE)}")

Found 20 files:
  LM tables_Q1_2021_AR_(%).xlsx
  LM tables_Q1_2022_AR_1 (%).xlsx
  LM tables_Q1_2023_AR(%).xlsx
  LM tables_Q1_2024_AR.xlsx
  LM tables_Q2_2021_AR_(%)).xlsx
  LM tables_Q2_2022_AR_(%).xlsx
  LM tables_Q2_2023_AR(%).xlsx
  LM tables_Q2_2024_AR.xlsx
  LM tables_Q3_2021_AR_1_(%).xlsx
  LM tables_Q3_2022_AR_1_(%).xlsx
  LM tables_Q3_2023_AR (%)_0.xlsx
  LM tables_Q4_2021_AR_1.xlsx
  LM tables_Q4_2022_AR_1_(%).xlsx
  LM tables_Q4_2023_AR (%)_0.xlsx
  Labor Market Statistics Q1 2025 - AR (1).xlsx
  Labor Market Statistics Q2 2025 AR.xlsx
  Labor Market Statistics Q3 2024 AR (2).xlsx
  Labor Market Statistics Q3_2025 AR.xlsx
  Labor Market Statistics Q4 2024 - AR (1).xlsx
  Labor Market Statistics Q4 2025 AR (1).xlsx

Latest file: Labor Market Statistics Q4 2025 AR (1).xlsx


In [17]:
DATA_COLS = [
    'saudi_male', 'saudi_female', 'saudi_total',
    'nonsaudi_male', 'nonsaudi_female', 'nonsaudi_total',
    'total_male', 'total_female', 'total_total',
]

ARABIC_QUARTER_MAP = {'الأول': 1, 'الثاني': 2, 'الثالث': 3, 'الرابع': 4}

def parse_quarter_year(filename):
    """Parse (quarter, year) from filename. Returns (None, None) on failure."""
    m = re.search(r'Q(\d)[\W_]+(\d{4})', os.path.basename(filename), re.IGNORECASE)
    if m:
        return int(m.group(1)), int(m.group(2))
    print(f"WARNING: Could not parse quarter/year from '{os.path.basename(filename)}'")
    return None, None

def parse_arabic_quarter_label(label):
    """'الربع الأول 2021' → (2021, 1). Returns (None, None) on failure."""
    s  = str(label).strip()
    ym = re.search(r'(\d{4})', s)
    year    = int(ym.group(1)) if ym else None
    quarter = next((q for w, q in ARABIC_QUARTER_MAP.items() if w in s), None)
    return year, quarter

def extract_cross_sectional(filepath, sheet_name, label_col):
    """
    Generic extractor for cross-sectional sheets.
    Col 0 = category label, cols 1-9 = nationality x gender data.
    Stops at and includes 'الإجمالي' (total row).
    """
    quarter, year = parse_quarter_year(filepath)
    if quarter is None:
        return pd.DataFrame()

    xl = pd.ExcelFile(filepath, engine='openpyxl')
    if sheet_name not in xl.sheet_names:
        print(f"WARNING: Sheet '{sheet_name}' not found — {os.path.basename(filepath)}")
        return pd.DataFrame()

    df      = pd.read_excel(xl, sheet_name=sheet_name, header=None)
    records = []

    for _, row in df.iterrows():
        label = str(row.iloc[0]).strip() if pd.notna(row.iloc[0]) else ''
        if not label or label.lower() == 'nan':
            continue
        try:
            float(row.iloc[1])
        except (TypeError, ValueError):
            continue
        records.append({'year': year, 'quarter': quarter, label_col: label,
                        **dict(zip(DATA_COLS, row.iloc[1:10].tolist()))})
        if label == 'الإجمالي':
            break

    return pd.DataFrame(records)

def find_sheet_by_header(filepath, header_label):
    """Find which sheet contains a specific Arabic header in col 0."""
    xl = pd.ExcelFile(filepath, engine='openpyxl')
    for sheet in xl.sheet_names:
        try:
            df = pd.read_excel(xl, sheet_name=sheet, header=None, nrows=7)
            col0 = df.iloc[:, 0].astype(str).str.strip()
            if col0.isin([header_label]).any():
                return sheet
        except:
            continue
    return None

# Smoke tests
assert parse_quarter_year('LM tables_Q1_2021_AR_(%).xlsx')               == (1, 2021)
assert parse_quarter_year('Labor Market Statistics Q3_2025 AR.xlsx')     == (3, 2025)
assert parse_quarter_year('Labor Market Statistics Q4 2025 AR (1).xlsx') == (4, 2025)
assert parse_arabic_quarter_label('الربع الأول 2021')                    == (2021, 1)
assert parse_arabic_quarter_label('الربع الرابع 2024')                   == (2024, 4)
assert parse_arabic_quarter_label(' الربع الرابع 2025   ')               == (2025, 4)
assert parse_arabic_quarter_label('الربع / السنة')                       == (None, None)
print("Helper functions OK ✓")

Helper functions OK ✓


In [18]:
SHEET1_INDICATORS = {
    'معدل البطالة':                        'unemployment_rate',
    'معدل المشتغلين من السكان في سن العمل': 'employment_rate',
    'معدل المشاركة في القوى العاملة':       'participation_rate',
}

def extract_main_indicators(filepath):
    """
    Extract 3 rates from Sheet '1' + avg_monthly_wage from Sheet '6-1'.

    Sheet 6-1 layouts:
      A (most files): target row = 'الإجمالي'           → data in cols 1-9
      B (2023 files): target row = 'السكان في سن العمل' → data in cols 2-10
    """
    quarter, year = parse_quarter_year(filepath)
    if quarter is None:
        return []

    records = []
    xl      = pd.ExcelFile(filepath, engine='openpyxl')
    sheets  = xl.sheet_names

    # Sheet '1' ──────────────────────────────────────────────────────────────
    if '1' not in sheets:
        print(f"WARNING: Sheet '1' missing — Q{quarter} {year}")
    else:
        df   = pd.read_excel(xl, sheet_name='1', header=None)
        col0 = df.iloc[:, 0].astype(str).str.strip()
        for arabic, indicator in SHEET1_INDICATORS.items():
            mask = col0 == arabic
            if not mask.any():
                print(f"WARNING: '{arabic}' not found — Q{quarter} {year}")
                continue
            row = df[mask].iloc[0]
            records.append({'year': year, 'quarter': quarter, 'indicator': indicator,
                            **dict(zip(DATA_COLS, row.iloc[1:10].tolist()))})

    # Sheet '6-1' ─────────────────────────────────────────────────────────────
    if '6-1' not in sheets:
        print(f"WARNING: Sheet '6-1' missing — Q{quarter} {year}")
    else:
        df6  = pd.read_excel(xl, sheet_name='6-1', header=None)
        col0 = df6.iloc[:, 0].astype(str).str.strip()

        mask_a = col0 == 'الإجمالي'
        if mask_a.any():                          # Layout A
            row    = df6[mask_a].iloc[-1]
            values = row.iloc[1:10].tolist()
        else:
            mask_b = col0 == 'السكان في سن العمل'
            if mask_b.any():                      # Layout B (2023)
                row    = df6[mask_b].iloc[-1]
                values = row.iloc[2:11].tolist()
            else:
                print(f"WARNING: No wage row in Sheet '6-1' — Q{quarter} {year}")
                values = None

        if values is not None:
            records.append({'year': year, 'quarter': quarter,
                            'indicator': 'avg_monthly_wage',
                            **dict(zip(DATA_COLS, values))})

    return records

# Quick sanity check
_s = extract_main_indicators(all_files[0])
print(f"Sample ({os.path.basename(all_files[0])}): {len(_s)} records")
for r in _s:
    print(f"  {r['indicator']:40s}  total_total={r['total_total']:.4f}")

Sample (LM tables_Q1_2021_AR_(%).xlsx): 4 records
  unemployment_rate                         total_total=5.8040
  employment_rate                           total_total=61.0566
  participation_rate                        total_total=64.8186
  avg_monthly_wage                          total_total=5745.4000


In [19]:
all_records = []
for fp in all_files:
    q, y = parse_quarter_year(fp)
    recs = extract_main_indicators(fp)
    all_records.extend(recs)
    print(f"Q{q} {y}  →  {len(recs)} records")

df_main = (pd.DataFrame(all_records)
             .sort_values(['year', 'quarter', 'indicator'])
             .reset_index(drop=True))

out = os.path.join(OUTPUT_DIR, 'combined_main.csv')
df_main.to_csv(out, index=False, encoding='utf-8-sig')
print(f"\nSaved {len(df_main)} rows → {out}")
df_main.head(8)

Q1 2021  →  4 records
Q1 2022  →  4 records
Q1 2023  →  4 records
Q1 2024  →  4 records
Q2 2021  →  4 records
Q2 2022  →  4 records
Q2 2023  →  4 records
Q2 2024  →  4 records
Q3 2021  →  4 records
Q3 2022  →  4 records
Q3 2023  →  4 records
Q4 2021  →  4 records
Q4 2022  →  4 records
Q4 2023  →  4 records
Q1 2025  →  4 records
Q2 2025  →  4 records
Q3 2024  →  4 records
Q3 2025  →  4 records
Q4 2024  →  4 records
Q4 2025  →  4 records

Saved 80 rows → C:\Users\Ragha\Projects\gastat-labor-market\output\combined_main.csv


,year,quarter,indicator,saudi_male,saudi_female,saudi_total,nonsaudi_male,nonsaudi_female,nonsaudi_total,total_male,total_female,total_total
0,2021,1,avg_monthly_wage,10968.500000,8445.200000,10403.900000,4104.600000,1937.900000,3892.000000,5855.600000,5032.900000,5745.400000
1,2021,1,employment_rate,59.970017,24.950974,42.406293,92.128778,36.473583,80.453113,79.389300,28.255962,61.056553
2,2021,1,participation_rate,64.788388,31.874167,48.280332,93.559902,38.545536,82.018674,82.162259,33.787691,64.818609
3,2021,1,unemployment_rate,7.437090,21.720389,12.166527,1.529634,5.375339,1.908786,3.374979,16.372027,5.803976
4,2021,2,avg_monthly_wage,10879.700000,8330.000000,10314.500000,4171.600000,2039.900000,3953.200000,6011.100000,5092.100000,5883.700000
5,2021,2,employment_rate,60.815109,24.842389,42.771250,90.708167,35.891755,78.848749,78.532525,27.994687,60.111813
6,2021,2,participation_rate,64.963181,32.125619,48.491912,92.774806,38.458432,81.023570,81.446945,33.932321,64.128176
7,2021,2,unemployment_rate,6.385266,22.671096,11.797147,2.227586,6.673899,2.684183,3.578305,17.498462,6.263023


In [20]:
# Sheet 3-6: % distribution of employed by sector x nationality x gender
# Sub-table 3-6-2 is the time series (all quarters stacked).
# Sector columns confirmed from Q4 2025 file header row:
#   الإجمالي (total)  → cols 1-9
#   قطاع عام (public) → cols 10-18
#   قطاع خاص (private)→ cols 19-27
#   اخرى (other)      → cols 28-36
#   غير مبين (unknown)→ cols 37-45

_SECTOR_FIXED = {'total': 1, 'public': 10, 'private': 19, 'other': 28, 'unknown': 37}

def _detect_sector_cols(df):
    AR_TO_ENG = {
        'الإجمالي': 'total', 'قطاع عام': 'public', 'القطاع الحكومي': 'public',
        'قطاع خاص': 'private', 'القطاع الخاص': 'private',
        'اخرى': 'other', 'أخرى': 'other', 'غير مبين': 'unknown',
    }
    col0 = df.iloc[:, 0].astype(str).str.strip()
    st2  = df.index[col0.str.contains('3-6-2', na=False)]
    if not len(st2):
        return _SECTOR_FIXED
    subtable_start = int(st2[0])
    first_data = next(
        (ri for ri in df.index
         if ri > subtable_start
         and parse_arabic_quarter_label(df.iloc[ri, 0])[0] is not None), None)
    if first_data is None:
        return _SECTOR_FIXED
    found = {}
    for ri in range(subtable_start + 1, int(first_data)):
        for ci, val in enumerate(df.iloc[ri]):
            if ci == 0:
                continue
            vs = str(val).strip()
            if vs in AR_TO_ENG and AR_TO_ENG[vs] not in found:
                found[AR_TO_ENG[vs]] = ci
    return found if found else _SECTOR_FIXED

def extract_sector_employment(filepath):
    xl = pd.ExcelFile(filepath, engine='openpyxl')
    if '3-6' not in xl.sheet_names:
        print(f"WARNING: Sheet '3-6' not found — {os.path.basename(filepath)}")
        return pd.DataFrame()
    df         = pd.read_excel(xl, sheet_name='3-6', header=None)
    sector_map = _detect_sector_cols(df)
    col0       = df.iloc[:, 0].astype(str).str.strip()
    src_rows   = df.index[col0.str.startswith('المصدر')]
    max_row    = src_rows[0] if len(src_rows) else len(df)
    records    = []
    for ri in df.index:
        if ri >= max_row:
            break
        yr, qr = parse_arabic_quarter_label(df.iloc[ri, 0])
        if yr is None:
            continue
        row = df.iloc[ri]
        for sec_name, start_col in sector_map.items():
            vals = [row.iloc[c] if c < len(row) else None
                    for c in range(start_col, start_col + 9)]
            records.append({'year': yr, 'quarter': qr, 'sector': sec_name,
                            **dict(zip(DATA_COLS, vals))})
    return pd.DataFrame(records)

df_sector = extract_sector_employment(LATEST_FILE)
df_sector = df_sector.sort_values(['year', 'quarter', 'sector']).reset_index(drop=True)
out = os.path.join(OUTPUT_DIR, 'sector_employment.csv')
df_sector.to_csv(out, index=False, encoding='utf-8-sig')
print(f"Saved {len(df_sector)} rows → {out}")
df_sector.head(10)

Saved 100 rows → C:\Users\Ragha\Projects\gastat-labor-market\output\sector_employment.csv


,year,quarter,sector,saudi_male,saudi_female,saudi_total,nonsaudi_male,nonsaudi_female,nonsaudi_total,total_male,total_female,total_total
0,2021,1,other,0.562166,0.554711,0.559966,29.977967,85.219764,35.231836,21.175483,31.901391,22.955136
1,2021,1,private,40.274450,60.848054,46.345725,66.768180,11.117197,61.475394,58.840107,42.435531,56.118244
2,2021,1,public,59.163384,38.597235,53.094309,3.253854,3.663039,3.292770,19.984410,25.663078,20.926620
3,2021,1,total,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
4,2021,1,unknown,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,2021,2,other,0.677540,1.159620,0.817988,28.143820,83.664965,33.611601,19.480521,31.337658,21.493251
6,2021,2,private,39.659730,59.376005,45.403836,68.463444,12.397552,62.942016,59.378297,42.192663,56.461064
7,2021,2,public,59.662730,39.464375,53.778176,3.392736,3.937483,3.446383,21.141181,26.469679,22.045685
8,2021,2,total,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
9,2021,2,unknown,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [21]:
# Sheet 2-11: % of Saudi unemployed willing to work in private sector.
# Time series starting Q1 2022. Col 0 = quarter label, cols 1-3 = Male, Female, Total.

def extract_private_sector_willingness(filepath):
    xl = pd.ExcelFile(filepath, engine='openpyxl')
    if '2-11' not in xl.sheet_names:
        print(f"WARNING: Sheet '2-11' not found — {os.path.basename(filepath)}")
        return pd.DataFrame()
    df       = pd.read_excel(xl, sheet_name='2-11', header=None)
    col0     = df.iloc[:, 0].astype(str).str.strip()
    src_rows = df.index[col0.str.startswith('المصدر')]
    max_row  = src_rows[0] if len(src_rows) else len(df)
    records  = []
    for ri in df.index:
        if ri >= max_row:
            break
        yr, qr = parse_arabic_quarter_label(df.iloc[ri, 0])
        if yr is None:
            continue
        row = df.iloc[ri]
        records.append({'year': yr, 'quarter': qr,
                        'male':   row.iloc[1],
                        'female': row.iloc[2],
                        'total':  row.iloc[3]})
    return pd.DataFrame(records)

df_willing = extract_private_sector_willingness(LATEST_FILE)
df_willing = df_willing.sort_values(['year', 'quarter']).reset_index(drop=True)
out = os.path.join(OUTPUT_DIR, 'private_sector_willingness.csv')
df_willing.to_csv(out, index=False, encoding='utf-8-sig')
print(f"Saved {len(df_willing)} rows → {out}")
df_willing

Saved 16 rows → C:\Users\Ragha\Projects\gastat-labor-market\output\private_sector_willingness.csv


,year,quarter,male,female,total
0,2022,1,95.089627,92.014782,93.030354
1,2022,2,96.144402,92.823627,93.896821
2,2022,3,95.458454,92.712302,93.477505
3,2022,4,95.036545,93.741127,94.188105
4,2023,1,95.359804,94.109026,94.552164
5,2023,2,95.902317,95.069229,95.370500
6,2023,3,96.250807,93.969501,94.772362
7,2023,4,96.579137,93.890330,94.938112
8,2024,1,97.345128,95.144131,95.946728
9,2024,2,96.108083,95.126773,95.485325


In [22]:
# Sheet 6-4: average working hours by nationality x gender (time series).
# Col 0 = quarter label, cols 1-9 = nationality x gender.

def extract_working_hours(filepath):
    xl = pd.ExcelFile(filepath, engine='openpyxl')
    if '6-4' not in xl.sheet_names:
        print(f"WARNING: Sheet '6-4' not found — {os.path.basename(filepath)}")
        return pd.DataFrame()
    df       = pd.read_excel(xl, sheet_name='6-4', header=None)
    col0     = df.iloc[:, 0].astype(str).str.strip()
    src_rows = df.index[col0.str.startswith('المصدر')]
    max_row  = src_rows[0] if len(src_rows) else len(df)
    records  = []
    for ri in df.index:
        if ri >= max_row:
            break
        yr, qr = parse_arabic_quarter_label(df.iloc[ri, 0])
        if yr is None:
            continue
        row = df.iloc[ri]
        records.append({'year': yr, 'quarter': qr,
                        **dict(zip(DATA_COLS, row.iloc[1:10].tolist()))})
    return pd.DataFrame(records)

df_hours = extract_working_hours(LATEST_FILE)
df_hours = df_hours.sort_values(['year', 'quarter']).reset_index(drop=True)
out = os.path.join(OUTPUT_DIR, 'working_hours.csv')
df_hours.to_csv(out, index=False, encoding='utf-8-sig')
print(f"Saved {len(df_hours)} rows → {out}")
df_hours

Saved 20 rows → C:\Users\Ragha\Projects\gastat-labor-market\output\working_hours.csv


,year,quarter,saudi_male,saudi_female,saudi_total,nonsaudi_male,nonsaudi_female,nonsaudi_total,total_male,total_female,total_total
0,2021,1,39.444470,36.743501,38.797392,45.778557,46.134477,45.812462,44.061640,41.188615,43.660782
1,2021,2,40.353965,36.647879,39.451247,46.407028,46.677740,46.433982,44.638295,41.202317,44.133138
2,2021,3,42.166855,37.359101,40.889258,47.641765,49.819784,47.876136,45.999023,42.808577,45.482898
3,2021,4,41.992835,37.847035,40.904193,49.355329,50.239615,49.432802,47.184464,42.705626,46.524853
4,2022,1,41.992530,38.186383,40.978698,49.240042,49.428883,49.256246,47.161901,42.586601,46.493573
5,2022,2,41.419169,38.687329,40.718280,48.189736,49.439767,48.289110,46.332246,42.969868,45.875112
6,2022,3,41.355964,39.069473,40.779716,47.863311,48.663143,47.927293,46.101613,43.003676,45.686737
7,2022,4,41.173464,38.842312,40.559968,47.539832,48.006205,47.575776,45.802753,42.360202,45.332591
8,2023,1,41.542493,38.511894,40.735567,48.220639,48.396852,48.235173,46.439128,42.515945,45.890103
9,2023,2,41.696705,39.061644,41.018011,48.067914,48.615387,48.110575,46.391414,42.935933,45.931432


In [23]:
# Sheet 3-3: % distribution of employed by occupation x nationality x gender.
# Filtered to 11 known occupation labels to exclude education sub-table
# that appears in newer files.

OCCUPATION_LABELS = [
    'المديرون',
    'الاختصاصيّون',
    'الفنيّون والاختصاصيّون المساعدون',
    'عاملو الدعم المكتبي',
    'عاملو الخدمات والمبيعات',
    'العاملون المهرة في الزراعة والغابات ومزارع الأسماك',
    'عاملو الحرف ومن يرتبط بهم',
    'مشغّلو المصانع والآلات وعاملو التجميع',
    'المهن الأولية',
    'غير مبين',
    'الإجمالي',
]

occ_dfs = []
for fp in all_files:
    q, y   = parse_quarter_year(fp)
    occ_sheet = find_sheet_by_header(fp, 'المهن')
    if occ_sheet is None:
        print(f"Q{q} {y}: ⚠ occupation sheet not found")
        continue
    df_occ = extract_cross_sectional(fp, occ_sheet, 'occupation')
    if not df_occ.empty:
        df_occ = df_occ[df_occ['occupation'].isin(OCCUPATION_LABELS)].reset_index(drop=True)
        occ_dfs.append(df_occ)
        print(f"Q{q} {y}: {len(df_occ)} rows (sheet {occ_sheet})")
    else:
        print(f"Q{q} {y}: ⚠ skipped")

df_occ_all = (pd.concat(occ_dfs, ignore_index=True)
                .sort_values(['year', 'quarter', 'occupation'])
                .reset_index(drop=True))
out = os.path.join(OUTPUT_DIR, 'occupation_employment.csv')
df_occ_all.to_csv(out, index=False, encoding='utf-8-sig')
print(f"\nSaved {len(df_occ_all)} rows → {out}")
df_occ_all.head(11)

Q1 2021: 11 rows (sheet 3-3)
Q1 2022: 11 rows (sheet 3-4)
Q1 2023: 11 rows (sheet 3-4)
Q1 2024: 11 rows (sheet 3-4)
Q2 2021: 11 rows (sheet 3-4)
Q2 2022: 11 rows (sheet 3-4)
Q2 2023: 11 rows (sheet 3-4)
Q2 2024: 11 rows (sheet 3-4)
Q3 2021: 11 rows (sheet 3-4)
Q3 2022: 11 rows (sheet 3-4)
Q3 2023: 11 rows (sheet 3-4)
Q4 2021: 11 rows (sheet 3-4)
Q4 2022: 11 rows (sheet 3-4)
Q4 2023: 11 rows (sheet 3-4)
Q1 2025: 11 rows (sheet 3-4)
Q2 2025: 11 rows (sheet 3-4)
Q3 2024: 11 rows (sheet 3-4)
Q3 2025: 11 rows (sheet 3-4)
Q4 2024: 11 rows (sheet 3-4)
Q4 2025: 11 rows (sheet 3-4)

Saved 220 rows → C:\Users\Ragha\Projects\gastat-labor-market\output\occupation_employment.csv


,year,quarter,occupation,saudi_male,saudi_female,saudi_total,nonsaudi_male,nonsaudi_female,nonsaudi_total,total_male,total_female,total_total
0,2021,1,الإجمالي,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
1,2021,1,الاختصاصيّون,21.355804,33.086727,24.817602,17.448009,7.942664,16.543986,18.617391,23.777303,19.473528
2,2021,1,العاملون المهرة في الزراعة والغابات ومزارع الأ...,0.971584,0.194105,0.742149,3.376001,0.054662,3.060119,2.656495,0.142477,2.239367
3,2021,1,الفنيّون والاختصاصيّون المساعدون,25.018637,9.863152,20.546250,11.048564,1.741126,10.163363,15.229016,6.856025,13.839761
4,2021,1,المديرون,8.297334,5.902845,7.590720,3.183654,0.582754,2.936291,4.713889,3.933116,4.584342
5,2021,1,المهن الأولية,2.206491,2.420932,2.269773,11.082285,43.160701,14.133160,8.426262,17.504564,9.932543
6,2021,1,عاملو الحرف ومن يرتبط بهم,1.463507,0.741915,1.250565,10.114214,0.322333,9.182940,7.525547,0.586567,6.374225
7,2021,1,عاملو الخدمات والمبيعات,16.844964,9.440709,14.659966,11.733636,44.258888,14.827008,13.263167,22.331911,14.767861
8,2021,1,عاملو الدعم المكتبي,11.370357,21.202489,14.271821,1.587527,1.065159,1.537847,4.514974,13.746775,6.046723
9,2021,1,غير مبين,9.011060,16.754586,11.296176,0.068603,0.351102,0.095470,2.744574,10.681304,4.061443


In [24]:
# Sheet 3-4: % distribution of employed by economic activity x nationality x gender.
# Filtered to 23 known activity labels to exclude occupation rows
# that appear in newer files.

ACTIVITY_LABELS = [
    'الزراعة والحراجة وصيد الأسماك',
    'التعدين واستغلال المحاجر',
    'الصناعة التحويلية',
    'إمدادات الكهرباء والغاز والبخار وتكييف الهواء',
    'إمدادات المياه وأنشطة الصرف الصحي وإدارة النفايات ومعالجتها',
    'التشييد',
    'تجارة الجملة والتجزئة، وإصلاح المركبات ذات المحركات والدراجات النارية',
    'النقل والتخزين',
    'أنشطة خدمات الإقامة والطعام',
    'المعلومات والاتصالات',
    'الأنشطة المالية وأنشطة التأمين',
    'الأنشطة العقارية',
    'الأنشطة المهنية والعلمية والتقنية',
    'أنشطة الخدمات الإدارية وخدمات الدعم',
    'الإدارة العامة والدفاع ، الضمان الاجتماعي الإلزامي',
    'التعليم',
    'الأنشطة في مجال صحة الإنسان والعمل الاجتماعي',
    'الفنون والترفيه والتسلية',
    'أنشطة الخدمات الأخرى',
    'أنشطة الأُسر المعيشية',
    'أنشطة المنظمات والهيئات غير الخاضعة للولاية القضائية الوطنية',
    'غير مبين',
    'الإجمالي',
]

eco_dfs = []
for fp in all_files:
    q, y   = parse_quarter_year(fp)
    eco_sheet = find_sheet_by_header(fp, 'الانشطة الاقتصادية')
    if eco_sheet is None:
        print(f"Q{q} {y}: ⚠ economic activity sheet not found")
        continue
    df_eco = extract_cross_sectional(fp, eco_sheet, 'economic_activity')
    if not df_eco.empty:
        df_eco = df_eco[df_eco['economic_activity'].isin(ACTIVITY_LABELS)].reset_index(drop=True)
        eco_dfs.append(df_eco)
        print(f"Q{q} {y}: {len(df_eco)} rows (sheet {eco_sheet})")
    else:
        print(f"Q{q} {y}: ⚠ skipped")

df_eco_all = (pd.concat(eco_dfs, ignore_index=True)
                .sort_values(['year', 'quarter', 'economic_activity'])
                .reset_index(drop=True))
out = os.path.join(OUTPUT_DIR, 'economic_activity_employment.csv')
df_eco_all.to_csv(out, index=False, encoding='utf-8-sig')
print(f"\nSaved {len(df_eco_all)} rows → {out}")
df_eco_all.head()

Q1 2021: 23 rows (sheet 3-4)
Q1 2022: 23 rows (sheet 3-5)
Q1 2023: 23 rows (sheet 3-5)
Q1 2024: 23 rows (sheet 3-5)
Q2 2021: 23 rows (sheet 3-5)
Q2 2022: 23 rows (sheet 3-5)
Q2 2023: 23 rows (sheet 3-5)
Q2 2024: 23 rows (sheet 3-5)
Q3 2021: 23 rows (sheet 3-5)
Q3 2022: 23 rows (sheet 3-5)
Q3 2023: 23 rows (sheet 3-5)
Q4 2021: 23 rows (sheet 3-5)
Q4 2022: 23 rows (sheet 3-5)
Q4 2023: 23 rows (sheet 3-5)
Q1 2025: 23 rows (sheet 3-5)
Q2 2025: 23 rows (sheet 3-5)
Q3 2024: 23 rows (sheet 3-5)
Q3 2025: 23 rows (sheet 3-5)
Q4 2024: 23 rows (sheet 3-5)
Q4 2025: 23 rows (sheet 3-5)

Saved 460 rows → C:\Users\Ragha\Projects\gastat-labor-market\output\economic_activity_employment.csv


,year,quarter,economic_activity,saudi_male,saudi_female,saudi_total,nonsaudi_male,nonsaudi_female,nonsaudi_total,total_male,total_female,total_total
0,2021,1,أنشطة الأُسر المعيشية,0.018014,0.106152,0.044023,29.549030,84.063578,34.733733,20.712069,31.190838,22.450716
1,2021,1,أنشطة الخدمات الأخرى,2.426707,2.388660,2.415479,1.582048,1.066885,1.533053,1.834807,1.899282,1.845505
2,2021,1,أنشطة الخدمات الإدارية وخدمات الدعم,3.226593,4.060777,3.472761,2.879695,1.130869,2.713370,2.983502,2.975997,2.982257
3,2021,1,أنشطة المنظمات والهيئات غير الخاضعة للولاية ال...,0.031184,0.000000,0.021982,0.068071,0.058229,0.067135,0.057033,0.021559,0.051147
4,2021,1,أنشطة خدمات الإقامة والطعام,2.113590,4.526723,2.825706,4.323099,0.246381,3.935375,3.661918,2.941955,3.542461


In [25]:
# Sheet 6-3: average monthly wage (SAR) by broad economic sector x nationality x gender.
# Only available in 2024+ files — pre-2024 sheet 6-3 contained working hours, not wages.
# Three sectors: الزراعة (Agriculture), الصناعة (Industry), الخدمات (Services).

WAGE_SECTOR_LABELS = ['الزراعة', 'الصناعة', 'الخدمات', 'غير مبين', 'الإجمالي']

wages_dfs = []
for fp in all_files:
    q, y = parse_quarter_year(fp)
    if y is None or y < 2024:
        continue  # Pre-2024 sheet 6-3 is working hours, not wages
    df_w = extract_cross_sectional(fp, '6-3', 'economic_activity')
    if not df_w.empty:
        df_w = df_w[df_w['economic_activity'].isin(WAGE_SECTOR_LABELS)].reset_index(drop=True)
        wages_dfs.append(df_w)
        print(f"Q{q} {y}: {len(df_w)} rows")
    else:
        print(f"Q{q} {y}: ⚠ no data")

df_wages = (pd.concat(wages_dfs, ignore_index=True)
              .sort_values(['year', 'quarter', 'economic_activity'])
              .drop_duplicates(subset=['year', 'quarter', 'economic_activity'])
              .reset_index(drop=True))
out = os.path.join(OUTPUT_DIR, 'wages_by_sector.csv')
df_wages.to_csv(out, index=False, encoding='utf-8-sig')
print(f"\nSaved {len(df_wages)} rows → {out}")
df_wages

Q1 2024: 5 rows
Q2 2024: 5 rows
Q1 2025: 4 rows
Q2 2025: 5 rows
Q3 2024: 5 rows
Q3 2025: 4 rows
Q4 2024: 5 rows
Q4 2025: 5 rows

Saved 38 rows → C:\Users\Ragha\Projects\gastat-labor-market\output\wages_by_sector.csv


,year,quarter,economic_activity,saudi_male,saudi_female,saudi_total,nonsaudi_male,nonsaudi_female,nonsaudi_total,total_male,total_female,total_total
0,2024,1,الإجمالي,11089.400000,7661.700000,10080.700000,4543.300000,3279.700000,4453.400000,6253.400000,6163.200000,6240.600000
1,2024,1,الخدمات,11356.700000,8245.300000,10455.800000,4404.200000,3221.400000,4285.000000,6544.700000,6326.100000,6507.800000
2,2024,1,الزراعة,6874.300000,4327.300000,5976.100000,2079.100000,3884.400000,2086.300000,2185.500000,4218.900000,2217.500000
3,2024,1,الصناعة,9794.700000,4940.000000,8278.800000,5153.500000,7394.600000,5162.400000,5867.000000,5053.100000,5811.500000
4,2024,1,غير مبين,5347.000000,5119.700000,5232.400000,979.200000,1152.400000,997.700000,2549.100000,4432.100000,3126.600000
5,2024,2,الإجمالي,11045.400000,8075.500000,10159.300000,4455.300000,3337.600000,4375.900000,6115.200000,6425.900000,6159.000000
6,2024,2,الخدمات,11292.900000,8774.300000,10554.800000,4336.900000,3291.400000,4231.900000,6392.000000,6630.000000,6431.900000
7,2024,2,الزراعة,6843.300000,5095.100000,6153.400000,2172.200000,5370.800000,2190.900000,2298.600000,5162.500000,2363.900000
8,2024,2,الصناعة,9909.600000,4982.900000,8381.800000,4999.400000,7225.900000,5006.300000,5739.900000,5067.000000,5695.600000
9,2024,2,غير مبين,6667.300000,4590.600000,5467.700000,1893.100000,1461.800000,1863.500000,3520.200000,4294.900000,3783.500000


**Setup required:** create `data/supplementary/` inside the project root and place both files there:
- `labor-market-administrative-records-data.csv` — from [KAPSARC Data Portal](https://datasource.kapsarc.org/explore/assets/labor-market-administrative-records-data/) (GOSI administrative records, curated by KAPSARC)
- `GOSI_Data.zip` — directly from [GOSI Open Data](https://www.gosi.gov.sa/ar/StatisticsAndData/OpenedData/download)

In [26]:
import zipfile, io

GOSI_DIR     = r"C:\Users\Ragha\Projects\gastat-labor-market\data\supplementary"
KAPSARC_PATH = os.path.join(GOSI_DIR, 'labor-market-administrative-records-data.csv')
GOSI_ZIP     = os.path.join(GOSI_DIR, 'gosi.zip')

# ── KAPSARC: quarterly headcount of GOSI contributors by nationality ──────────
kapsarc = pd.read_csv(KAPSARC_PATH, sep=';', on_bad_lines='skip')

# Keep only the overall totals: all breakdown columns must be NaN
mask = (
    (kapsarc['Indicator'] == 'Participants on the job Subject to the rules and regulations of social insurance') &
    (kapsarc['Gender']      == 'Total') &
    (kapsarc['Nationality'].isin(['Saudi', 'Non Saudi', 'Total'])) &
    kapsarc['Adopted Regulation'].isna() &
    kapsarc['Sector'].isna() &
    kapsarc['Age Group'].isna() &
    kapsarc['Administrative Region'].isna() &
    kapsarc['Occupation'].isna() &
    kapsarc['Economic Activity'].isna() &
    kapsarc['Educational level'].isna() &
    kapsarc['Reason for discontinuation'].isna()
)
headcount = kapsarc[mask][['Year', 'Quarter', 'Nationality', 'Persons']].copy()
headcount.columns = ['year', 'quarter', 'nationality', 'persons']
headcount['year'] = headcount['year'].astype('Int64')
headcount = (
    headcount
    .dropna(subset=['year', 'quarter', 'persons'])
    .drop_duplicates(subset=['year', 'quarter', 'nationality'])
    .sort_values(['year', 'quarter', 'nationality'])
    .reset_index(drop=True)
)

out = os.path.join(OUTPUT_DIR, 'gosi_headcount.csv')
headcount.to_csv(out, index=False, encoding='utf-8-sig')
print(f"gosi_headcount: {len(headcount)} rows → {out}")
print(headcount.groupby(['year','nationality']).agg(persons=('persons','max')).unstack('nationality').tail(10).to_string())


gosi_headcount: 108 rows → C:\Users\Ragha\Projects\gastat-labor-market\output\gosi_headcount.csv
                persons                       
nationality   Non Saudi      Saudi       Total
year                                          
2017          8449330.0  1982155.0  10309339.0
2018          7733256.0  1972081.0   9705337.0
2019          6740395.0  1953770.0   8673507.0
2020          6725379.0  2027537.0   8700903.0
2021          6301792.0  2240812.0   8531308.0
2022          7341623.0  2581415.0   9923038.0
2023          8116191.0  2738430.0  10842733.0
2024          9544713.0  2893772.0  12438485.0
2025         10589566.0  3075549.0  13665115.0


In [27]:
# ── GOSI: Saudi/Non-Saudi split by sector (government vs private) ────────────
# Files: '*Percentage of Saudis & Non-Saudis by Sector*.xlsx' inside GOSI_Data.zip

sector_records = []
with zipfile.ZipFile(GOSI_ZIP) as z:
    for name in z.namelist():
        if 'Percentage' in name and name.endswith('.xlsx'):
            m = re.search(r'(\d{4})Q(\d)', name)
            if not m:
                continue
            yr, qr = int(m.group(1)), int(m.group(2))
            try:
                raw = pd.read_excel(io.BytesIO(z.read(name)), header=None)
            except Exception as e:
                print(f'WARNING: could not read {name}: {e}')
                continue
            for _, row in raw.iterrows():
                label = str(row.iloc[0]).strip()
                if 'حكومي' in label or 'خاص' in label:
                    sector_en = 'government' if 'حكومي' in label else 'private'
                    try:
                        s_pct = round(float(row.iloc[1]) * 100, 1)
                        n_pct = round(float(row.iloc[2]) * 100, 1)
                        sector_records.append({
                            'year': yr, 'quarter': qr,
                            'sector': sector_en,
                            'saudi_pct': s_pct,
                            'nonsaudi_pct': n_pct,
                        })
                    except (ValueError, TypeError, IndexError):
                        continue

df_gosi_sector = (
    pd.DataFrame(sector_records)
    .sort_values(['year', 'quarter', 'sector'])
    .drop_duplicates(subset=['year', 'quarter', 'sector'])
    .reset_index(drop=True)
)
out = os.path.join(OUTPUT_DIR, 'gosi_sector_split.csv')
df_gosi_sector.to_csv(out, index=False, encoding='utf-8-sig')
print(f"gosi_sector_split: {len(df_gosi_sector)} rows → {out}")
df_gosi_sector


gosi_sector_split: 10 rows → C:\Users\Ragha\Projects\gastat-labor-market\output\gosi_sector_split.csv


,year,quarter,sector,saudi_pct,nonsaudi_pct
0,2025,1,government,81.0,19.0
1,2025,1,private,19.9,80.1
2,2025,2,government,80.9,19.1
3,2025,2,private,19.7,80.3
4,2025,3,government,81.4,18.6
5,2025,3,private,19.8,80.2
6,2025,4,government,81.1,18.9
7,2025,4,private,19.6,80.4
8,2026,1,government,81.1,18.9
9,2026,1,private,19.6,80.5


In [28]:
OUTPUTS = {
    'combined_main.csv':                {'key': 'indicator'},
    'sector_employment.csv':            {'key': 'sector'},
    'private_sector_willingness.csv':   {'key': None},
    'working_hours.csv':                {'key': None},
    'occupation_employment.csv':        {'key': 'occupation'},
    'economic_activity_employment.csv': {'key': 'economic_activity'},
    'wages_by_sector.csv':              {'key': 'economic_activity'},
    'gosi_headcount.csv':              {'key': 'nationality'},
    'gosi_sector_split.csv':           {'key': 'sector'},
}

print(f"{'Output':<38}  {'Rows':>6}  {'Periods':>8}  {'Nulls':>6}  {'Categories'}")
print("-" * 80)
for fname, meta in OUTPUTS.items():
    fpath = os.path.join(OUTPUT_DIR, fname)
    if not os.path.exists(fpath):
        print(f"{'MISSING: ' + fname:<38}")
        continue
    df      = pd.read_csv(fpath, encoding='utf-8-sig')
    periods = df.groupby(['year','quarter']).ngroups if 'year' in df.columns else '-'
    nulls   = df.isnull().sum().sum()
    key     = meta['key']
    cats    = df[key].nunique() if key and key in df.columns else '-'
    print(f"{fname:<38}  {len(df):>6}  {str(periods):>8}  {nulls:>6}  {cats}")


Output                                    Rows   Periods   Nulls  Categories
--------------------------------------------------------------------------------
combined_main.csv                           80        20       0  4
sector_employment.csv                      100        20       0  5
private_sector_willingness.csv              16        16       0  -
working_hours.csv                           20        20       0  -
occupation_employment.csv                  220        20       0  11
economic_activity_employment.csv           460        20       0  23
wages_by_sector.csv                         38         8       0  5
gosi_headcount.csv                         108        36       0  3
gosi_sector_split.csv                       10         5       0  2
